In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q transformers datasets peft accelerate bitsandbytes pandas

In [ ]:
!ls -lh /content/

total 8.7M
drwx------ 5 root root 4.0K Oct 24 16:09 drive
-rw-r--r-- 1 root root 8.7M Oct 24 16:08 public_task1_train.jsonl
drwxr-xr-x 1 root root 4.0K Oct 21 13:28 sample_data
-rw-r--r-- 1 root root 9.9K Oct 24 16:12 train_qwen_lora.py


In [ ]:
!nvidia-smi
!pip install -q transformers datasets peft accelerate bitsandbytes pandas
!pip install -q --upgrade "torchao>=0.16.0"

Sat Oct 24 16:19:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -q --upgrade "torchao>=0.16.0"

In [ ]:
!python /content/train_qwen_lora.py \
  --train-file "/content/public_task1_train.jsonl" \
  --output-dir "/content/drive/MyDrive/output_model" \
  --no-bf16

Map: 100% 9141/9141 [00:24<00:00, 380.06 examples/s]
Map: 100% 483/483 [00:00<00:00, 529.24 examples/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 434/434 [00:16<00:00, 25.65it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 151643}.
{'loss': '1.353', 'grad_norm': '13.83', 'learning_rate': '3.429e-05', 'epoch': '0.0175'}
{'loss': '0.662', 'grad_norm': '0.8314', 'learning_rate': '9.143e-05', 'epoch': '0.035'}
{'loss': '0.4229', 'grad_norm': '2.017', 'learning_rate': '0.0001486', 'epoch': '0.0525'}
{'loss': '0.4148', 'grad_norm': '0.6919', 'learning_rate': '0.0002', 'epoch': '0.07001'}
{'loss': '0.4283', 'grad_norm': '1.073', 'learning_rate': '0.0002', 'epoch': '0.08751'}
{'loss': '0.4073', 'grad_norm': '1.343', 'learning_rate': '0.0001998

In [ ]:
"""Vyhodnotenie na validacnej casti public_task1_train.jsonl (split == 'validation')."""
import argparse, json, collections, os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ============================================================
# NASTAVENIA – uprav podľa svojho prostredia v Colabe
# ============================================================
DEFAULTS = {
    "model":   "Qwen/Qwen2.5-3B-Instruct",          # alebo "mistralai/Ministral-3-3B-Instruct-2512" ak máš adapter preň
    "adapter": "/content/drive/MyDrive/output_model/final_adapter",                                 # napr. "/content/drive/MyDrive/output_model/final_adapter"
    "data":    "/content/public_task1_train.jsonl",  # cesta k public_task1_train.jsonl v Colabe
    "split":   "validation",
    "out":     "/content/eval_result.json",
}

ap = argparse.ArgumentParser()
for k, v in DEFAULTS.items():
    ap.add_argument(f"--{k}", default=v)
a, _ = ap.parse_known_args()   # <- parse_known_args ignoruje Colab kernel args

# ---------- sanity check ----------
if not os.path.exists(a.data):
    raise FileNotFoundError(
        f"Nenájdem {a.data}. Nahraj public_task1_train.jsonl do /content "
        f"alebo uprav DEFAULTS['data']."
    )

# ---------- dtype: bf16 len na Ampere+ (A100/L4), T4 potrebuje fp16 ----------
dtype = torch.bfloat16 if (
    torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
) else torch.float16
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("dtype:", dtype)

# ---------- načítanie modelu ----------
tok = AutoTokenizer.from_pretrained(a.model)
model = AutoModelForCausalLM.from_pretrained(a.model, torch_dtype=dtype, device_map="auto")
if a.adapter:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, a.adapter)
model.eval()

letters = "ABCDEF"
ids = [tok.encode(l, add_special_tokens=False)[0] for l in letters]

# ---------- vyhodnotenie ----------
rows = [json.loads(l) for l in open(a.data)]
rows = [r for r in rows if r["split"] == a.split]
print(f"Validation examples: {len(rows)}")

res, stat = [], collections.defaultdict(lambda: [0, 0])
for i, r in enumerate(rows, 1):
    text = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True
    )
    x = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**x).logits[0, -1]
    n = r["num_choices"]
    pred = letters[int(torch.argmax(logits[ids[:n]]))]
    ok = pred == r["answer"].strip().upper()[:1]
    for k in ("ALL", r["language"], r["source"]):
        stat[k][0] += ok; stat[k][1] += 1
    res.append({"id": r["id"], "pred": pred, "gold": r["answer"], "ok": ok})
    if i % 50 == 0:
        print(f"  {i}/{len(rows)}")

summary = {k: {"acc": round(c / n, 4), "correct": c, "n": n} for k, (c, n) in sorted(stat.items())}
print(json.dumps(summary, indent=1))
json.dump({"summary": summary, "preds": res}, open(a.out, "w"), indent=1)
print(f"Výsledky uložené v {a.out}")

GPU: Tesla T4
dtype: torch.float16


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Validation examples: 483
  50/483
  100/483
  150/483
  200/483
  250/483
  300/483
  350/483
  400/483
  450/483
{
 "ALL": {
  "acc": 0.4928,
  "correct": 238,
  "n": 483
 },
 "arabic": {
  "acc": 0.5909,
  "correct": 13,
  "n": 22
 },
 "arabic_accounting_train": {
  "acc": 0.5714,
  "correct": 4,
  "n": 7
 },
 "arabic_business_train": {
  "acc": 0.6,
  "correct": 9,
  "n": 15
 },
 "bhashabench_en": {
  "acc": 0.5023,
  "correct": 107,
  "n": 213
 },
 "bhashabench_hi": {
  "acc": 0.4341,
  "correct": 89,
  "n": 205
 },
 "cfa_cpa_chinese": {
  "acc": 0.4,
  "correct": 2,
  "n": 5
 },
 "cfa_cpa_english": {
  "acc": 0.5909,
  "correct": 13,
  "n": 22
 },
 "chinese": {
  "acc": 0.4,
  "correct": 2,
  "n": 5
 },
 "english": {
  "acc": 0.5106,
  "correct": 120,
  "n": 235
 },
 "flare_es": {
  "acc": 0.9,
  "correct": 9,
  "n": 10
 },
 "greek": {
  "acc": 0.8333,
  "correct": 5,
  "n": 6
 },
 "hindi": {
  "acc": 0.4341,
  "correct": 89,
  "n": 205
 },
 "plutus": {
  "acc": 0.8333,
  "correct

In [ ]:
!pip install -q --upgrade "transformers>=5.0.0" "mistral-common>=1.8.6" accelerate peft

In [ ]:
import os
OUT_DIR = "/content/drive/MyDrive/CLEF_2026_ministral3_lora"
os.makedirs(OUT_DIR, exist_ok=True)
print("Výstup pôjde do:", OUT_DIR)

Výstup pôjde do: /content/drive/MyDrive/CLEF_2026_ministral3_lora


In [3]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!python /content/train_qwen_lora.py \
  --train-file /content/public_task1_train.jsonl \
  --output-dir /content/drive/MyDrive/CLEF_2026_ministral3_lora \
  --model-name mistralai/Ministral-3-3B-Instruct-2512 \
  --bf16 \
  --per-device-train-batch-size 2 \
  --gradient-accumulation-steps 8 \
  --num-train-epochs 2 \
  --logging-steps 10 \
  --eval-steps 100 \
  --save-steps 100

config.json: 100% 1.90k/1.90k [00:00<00:00, 5.37MB/s]
tokenizer_config.json: 100% 21.2k/21.2k [00:00<00:00, 50.0MB/s]

tokenizer.json: downloading bytes:   0% 0.00/17.1M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 4.48M/4.48M [00:00<00:00, 6.00MB/s,  440kB/s  ]
tokenizer.json: reconstructing file: 100% 17.1M/17.1M [00:00<00:00, 22.9MB/s, 1.68MB/s  ]
chat_template.jinja: 100% 11.9k/11.9k [00:00<00:00, 28.1MB/s]
Map: 100% 9141/9141 [00:40<00:00, 227.11 examples/s]
Map: 100% 483/483 [00:02<00:00, 217.84 examples/s]

model.safetensors: downloading bytes:   1% 69.7M/4.67G [00:01<00:44, 104MB/s, 5.03MB/s  ] 
model.safetensors: downloading bytes:   2% 85.7M/4.67G [00:01<00:48, 94.1MB/s, 7.21MB/s  ]
model.safetensors: reconstructing file:   2% 87.5M/4.67G [00:01<01:16, 59.9MB/s, 8.22MB/s  ]
model.safetensors: downloading bytes:   3% 129M/4.67G [00:01<00:49, 91.1MB/s, 10.8MB/s  ]
model.safetensors: reconstructing file:   3% 149M/4.67G [00:01<00:51, 87.2MB/s, 10.6MB/s  ]
model.safete

In [7]:
"""Vyhodnotenie Ministral 3 3B + LoRA na validacnej casti public_task1_train.jsonl."""
import argparse, json, collections, os, torch
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration  # ZMENA: import správnej triedy

# ============================================================
# NASTAVENIA
# ============================================================
DEFAULTS = {
    "model":   "mistralai/Ministral-3-3B-Instruct-2512",
    "adapter": "/content/drive/MyDrive/CLEF_2026_ministral3_lora/final_adapter",
    "data":    "/content/public_task1_train.jsonl",
    "split":   "validation",
    "out":     "/content/drive/MyDrive/CLEF_2026_ministral3_lora/eval_ministral3.json",
}

ap = argparse.ArgumentParser()
for k, v in DEFAULTS.items():
    ap.add_argument(f"--{k}", default=v)
a, _ = ap.parse_known_args()

# ---------- sanity check ----------
if not os.path.exists(a.data):
    raise FileNotFoundError(f"Nenájdem {a.data}")
if not os.path.exists(a.adapter):
    raise FileNotFoundError(
        f"Nenájdem adapter v {a.adapter}. "
        f"Skontroluj, či tréning dobehol a či tam je final_adapter/."
    )

# ---------- dtype: bf16 na L4 funguje ----------
dtype = torch.bfloat16 if (
    torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
) else torch.float16
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("dtype:", dtype)
print("Model:", a.model)
print("Adapter:", a.adapter)

# ---------- načítanie modelu ----------
tok = AutoTokenizer.from_pretrained(a.model)

# ZMENA: Použite Mistral3ForConditionalGeneration namiesto AutoModelForCausalLM
model = Mistral3ForConditionalGeneration.from_pretrained(
    a.model,
    dtype=dtype,  # ZMENA: 'dtype' namiesto 'torch_dtype' (odstráni varovanie)
    device_map="auto"
)

from peft import PeftModel
model = PeftModel.from_pretrained(model, a.adapter)
model.eval()

letters = "ABCDEF"
ids = [tok.encode(l, add_special_tokens=False)[0] for l in letters]

# ---------- vyhodnotenie ----------
rows = [json.loads(l) for l in open(a.data)]
rows = [r for r in rows if r["split"] == a.split]
print(f"Validation examples: {len(rows)}")

res, stat = [], collections.defaultdict(lambda: [0, 0])
for i, r in enumerate(rows, 1):
    text = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True
    )
    x = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**x).logits[0, -1]
    n = r["num_choices"]
    pred = letters[int(torch.argmax(logits[ids[:n]]))]
    ok = pred == r["answer"].strip().upper()[:1]
    for k in ("ALL", r["language"], r["source"]):
        stat[k][0] += ok; stat[k][1] += 1
    res.append({"id": r["id"], "pred": pred, "gold": r["answer"], "ok": ok})
    if i % 50 == 0:
        print(f"  {i}/{len(rows)}")

summary = {k: {"acc": round(c / n, 4), "correct": c, "n": n} for k, (c, n) in sorted(stat.items())}
print(json.dumps(summary, indent=1))
json.dump({"summary": summary, "preds": res}, open(a.out, "w"), indent=1)
print(f"Výsledky uložené v {a.out}")

GPU: NVIDIA A100-SXM4-80GB
dtype: torch.bfloat16
Model: mistralai/Ministral-3-3B-Instruct-2512
Adapter: /content/drive/MyDrive/CLEF_2026_ministral3_lora/final_adapter


[transformers] FP8 quantized models is only supported on GPUs with compute capability >= 8.9 (e.g 4090/H100), actual = `8.0`. We will default to dequantizing the model to bf16. Feel free to use a different quantization method like bitsandbytes or torchao


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.


Validation examples: 483


...

  400/483


[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=Tr

  450/483


[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=Tr

{
 "ALL": {
  "acc": 0.4824,
  "correct": 233,
  "n": 483
 },
 "arabic": {
  "acc": 0.5455,
  "correct": 12,
  "n": 22
 },
 "arabic_accounting_train": {
  "acc": 0.5714,
  "correct": 4,
  "n": 7
 },
 "arabic_business_train": {
  "acc": 0.5333,
  "correct": 8,
  "n": 15
 },
 "bhashabench_en": {
  "acc": 0.4695,
  "correct": 100,
  "n": 213
 },
 "bhashabench_hi": {
  "acc": 0.4488,
  "correct": 92,
  "n": 205
 },
 "cfa_cpa_chinese": {
  "acc": 0.2,
  "correct": 1,
  "n": 5
 },
 "cfa_cpa_english": {
  "acc": 0.5909,
  "correct": 13,
  "n": 22
 },
 "chinese": {
  "acc": 0.2,
  "correct": 1,
  "n": 5
 },
 "english": {
  "acc": 0.4809,
  "correct": 113,
  "n": 235
 },
 "flare_es": {
  "acc": 1.0,
  "correct": 10,
  "n": 10
 },
 "greek": {
  "acc": 0.8333,
  "correct": 5,
  "n": 6
 },
 "hindi": {
  "acc": 0.4488,
  "correct": 92,
  "n": 205
 },
 "plutus": {
  "acc": 0.8333,
  "correct": 5,
  "n": 6
 },
 "spanish": {
  "acc": 1.0,
  "correct": 10,
  "n": 10
 }
}
Výsledky uložené v /content/dri

In [9]:
DEFAULTS = {
    "model":  "mistralai/Ministral-3-3B-Instruct-2512",
    "data":   "/content/public_task1_train.jsonl",
    "split":  "validation",
    "out":    "/content/drive/MyDrive/CLEF_2026_ministral3_lora/eval_ministral3_baseHF.json",
}

ap = argparse.ArgumentParser()
for k, v in DEFAULTS.items():
    ap.add_argument(f"--{k}", default=v)
a, _ = ap.parse_known_args()

if not os.path.exists(a.data):
    raise FileNotFoundError(f"Nenájdem {a.data}")

dtype = torch.bfloat16 if (
    torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
) else torch.float16
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("dtype:", dtype)
print("Model:", a.model, "(baseline, bez adaptera)")

tok = AutoTokenizer.from_pretrained(a.model)
model = Mistral3ForConditionalGeneration.from_pretrained(
    a.model,
    dtype=dtype,
    device_map="auto",
)
model.eval()

letters = "ABCDEF"
ids = [tok.encode(l, add_special_tokens=False)[0] for l in letters]

rows = [json.loads(l) for l in open(a.data)]
rows = [r for r in rows if r["split"] == a.split]
print(f"Validation examples: {len(rows)}")

res, stat = [], collections.defaultdict(lambda: [0, 0])
for i, r in enumerate(rows, 1):
    text = tok.apply_chat_template(
        [{"role": "user", "content": r["prompt"]}],
        tokenize=False, add_generation_prompt=True
    )
    x = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**x).logits[0, -1]
    n = r["num_choices"]
    pred = letters[int(torch.argmax(logits[ids[:n]]))]
    ok = pred == r["answer"].strip().upper()[:1]
    for k in ("ALL", r["language"], r["source"]):
        stat[k][0] += ok; stat[k][1] += 1
    res.append({"id": r["id"], "pred": pred, "gold": r["answer"], "ok": ok})
    if i % 50 == 0:
        print(f"  {i}/{len(rows)}")

summary = {k: {"acc": round(c / n, 4), "correct": c, "n": n} for k, (c, n) in sorted(stat.items())}
print(json.dumps(summary, indent=1))
json.dump({"summary": summary, "preds": res}, open(a.out, "w"), indent=1)
print(f"Výsledky uložené v {a.out}")

GPU: NVIDIA A100-SXM4-80GB
dtype: torch.bfloat16
Model: mistralai/Ministral-3-3B-Instruct-2512 (baseline, bez adaptera)


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=Tr

Validation examples: 483


[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=Tr

...

  400/483


[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=Tr

  450/483


[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=True` instead and don't encode the output manually.
[transformers] `MistralCommonBackend.apply_chat_template(..., tokenize=False)` is unsafe and may lead to unexpected behavior. Please consider using `tokenize=Tr

{
 "ALL": {
  "acc": 0.412,
  "correct": 199,
  "n": 483
 },
 "arabic": {
  "acc": 0.5,
  "correct": 11,
  "n": 22
 },
 "arabic_accounting_train": {
  "acc": 0.5714,
  "correct": 4,
  "n": 7
 },
 "arabic_business_train": {
  "acc": 0.4667,
  "correct": 7,
  "n": 15
 },
 "bhashabench_en": {
  "acc": 0.446,
  "correct": 95,
  "n": 213
 },
 "bhashabench_hi": {
  "acc": 0.361,
  "correct": 74,
  "n": 205
 },
 "cfa_cpa_chinese": {
  "acc": 0.4,
  "correct": 2,
  "n": 5
 },
 "cfa_cpa_english": {
  "acc": 0.5,
  "correct": 11,
  "n": 22
 },
 "chinese": {
  "acc": 0.4,
  "correct": 2,
  "n": 5
 },
 "english": {
  "acc": 0.4511,
  "correct": 106,
  "n": 235
 },
 "flare_es": {
  "acc": 0.4,
  "correct": 4,
  "n": 10
 },
 "greek": {
  "acc": 0.3333,
  "correct": 2,
  "n": 6
 },
 "hindi": {
  "acc": 0.361,
  "correct": 74,
  "n": 205
 },
 "plutus": {
  "acc": 0.3333,
  "correct": 2,
  "n": 6
 },
 "spanish": {
  "acc": 0.4,
  "correct": 4,
  "n": 10
 }
}
Výsledky uložené v /content/drive/MyDrive/CL

In [1]:
"""Vyhodnotenie Ministral 3 3B + LoRA na test casti datasetu."""
import argparse, json, collections, os, torch
from transformers import AutoTokenizer, Mistral3ForConditionalGeneration

DEFAULTS = {
    "model":   "mistralai/Ministral-3-3B-Instruct-2512",
    "adapter": "/content/drive/MyDrive/CLEF_2026_ministral3_lora/final_adapter",
    "data":    "/content/hindi_econ_test.jsonl",
    "split":   "test",
    "out":     "/content/drive/MyDrive/CLEF_2026_ministral3_lora/eval_hindi_econ_test_odclauduupraven.json",
}
ap = argparse.ArgumentParser()
for k, v in DEFAULTS.items():
    ap.add_argument(f"--{k}", default=v)
a, _ = ap.parse_known_args()

if not os.path.exists(a.data):
    raise FileNotFoundError(f"Nenájdem {a.data}")
if not os.path.exists(a.adapter):
    raise FileNotFoundError(f"Nenájdem adapter v {a.adapter}.")

dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8) else torch.float16
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("dtype:", dtype, "| Model:", a.model, "| Adapter:", a.adapter)

tok = AutoTokenizer.from_pretrained(a.model)
model = Mistral3ForConditionalGeneration.from_pretrained(a.model, dtype=dtype, device_map="auto")
from peft import PeftModel
model = PeftModel.from_pretrained(model, a.adapter)
model.eval()

letters = "ABCDEF"
ids = [tok.encode(l, add_special_tokens=False)[0] for l in letters]

rows = [json.loads(l) for l in open(a.data, encoding="utf-8")]
rows = [r for r in rows if r["split"] == a.split]
print(f"Examples ({a.split}): {len(rows)}")

res, stat = [], collections.defaultdict(lambda: [0, 0])
for i, r in enumerate(rows, 1):
    text = tok.apply_chat_template([{"role": "user", "content": r["prompt"]}],
                                   tokenize=False, add_generation_prompt=True)
    x = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        logits = model(**x).logits[0, -1]
    n = r["num_choices"]
    pred = letters[int(torch.argmax(logits[ids[:n]]))]
    ok = pred == r["answer"].strip().upper()[:1]
    for k in ("ALL", r["language"], r["source"]):
        stat[k][0] += ok; stat[k][1] += 1
    res.append({"id": r["id"], "pred": pred, "gold": r["answer"], "ok": ok})
    if i % 50 == 0:
        print(f"  {i}/{len(rows)}")

summary = {k: {"acc": round(c / n, 4), "correct": c, "n": n} for k, (c, n) in sorted(stat.items())}
print(json.dumps(summary, indent=1))
os.makedirs(os.path.dirname(a.out), exist_ok=True)
json.dump({"summary": summary, "preds": res}, open(a.out, "w"), indent=1)
print(f"Výsledky uložené v {a.out}")

GPU: NVIDIA A100-SXM4-80GB
dtype: torch.bfloat16 | Model: mistralai/Ministral-3-3B-Instruct-2512 | Adapter: /content/drive/MyDrive/CLEF_2026_ministral3_lora/final_adapter


[transformers] The tokenizer you are loading from 'mistralai/Ministral-3-3B-Instruct-2512' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
[transformers] FP8 quantized models is only supported on GPUs with compute capability >= 8.9 (e.g 4090/H100), actual = `8.0`. We will default to dequantizing the model to bf16. Feel free to use a different quantization method like bitsandbytes or torchao


Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

Examples (test): 100
  50/100
  100/100
{
 "ALL": {
  "acc": 0.66,
  "correct": 66,
  "n": 100
 },
 "hindi": {
  "acc": 0.66,
  "correct": 66,
  "n": 100
 },
 "hindi_econ_dev": {
  "acc": 0.66,
  "correct": 66,
  "n": 100
 }
}
Výsledky uložené v /content/drive/MyDrive/CLEF_2026_ministral3_lora/eval_hindi_econ_test_odclauduupraven.json


In [2]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [1]:
"""
Unified eval: otestuje Qwen aj Ministral (baseline + LoRA) na test dátach s labelmi
a uloží výsledky + porovnanie na disk.

Predpokladaný formát test jsonl (rovnaký ako public_task1_train.jsonl):
  {"id": "...", "language": "hindi", "source": "bhashabench_hi",
   "prompt": "...", "answer": "B", "num_choices": 4, "split": "validation"}

Ak tvoj test súbor nemá "prompt", ale má "question" + "options" + "num_choices",
skript si prompt poskladá automaticky z LANGUAGE_PROMPTS nižšie.
"""
import json, os, collections, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, Mistral3ForConditionalGeneration

# ============================================================
# CONFIG – UPRAV LEN TOTO
# ============================================================
CONFIG = {
    # Cesty k adaptérom. None = baseline (bez LoRA).
    # Príklad s LoRA: "/content/drive/MyDrive/CLEF_2026_qwen_lora/final_adapter"
    "qwen_adapter":      "/content/drive/MyDrive/output_model_qwen_LoRA/final_adapter",
    "ministral_adapter": "/content/drive/MyDrive/CLEF_2026_ministral3_lora/final_adapter",

    # Test súbor s labelmi (musí mať answer/gold_answer a num_choices)
    "test_data": "/content/task1_final_test_public_data_label_by_LLM.jsonl",

    # Výstupný priečinok
    "out_dir": "/content/drive/MyDrive/CLEF_2026_results_LLMmix_labeled",

    # Split filter (None = ber všetko, alebo napr. "validation")
    "split": None,

    # Base modely (nemenné)
    "qwen_base":      "Qwen/Qwen2.5-3B-Instruct",
    "ministral_base": "mistralai/Ministral-3-3B-Instruct-2512",
}

os.makedirs(CONFIG["out_dir"], exist_ok=True)

# ============================================================
# POMOCNÉ
# ============================================================
LANGUAGE_PROMPTS = {
    "hindi":   "निम्नलिखित प्रश्न पढ़ें और सही उत्तर चुनें। केवल विकल्प का अक्षर लिखें।",
    "chinese": "请阅读下面的选择题，并只返回正确选项对应的字母。",
    "arabic":  "اقرأ السؤال التالي واختر الإجابة الصحيحة فقط. أجب بحرف الخيار فقط.",
    "english": "Read the following multiple-choice question and return only the correct option letter.",
    "spanish": "Lee la siguiente pregunta de opción múltiple y devuelve solo la letra de la respuesta correcta.",
    "greek":   "Διάβασε την παρακάτω ερώτηση πολλαπλής επιλογής και επέστρεψε μόνο το γράμμα της σωστής απάντησης.",
}

LETTERS = "ABCDEF"


def build_prompt_from_row(row):
    """Ak má row 'prompt', vráti ho. Inak poskladá z question + options + language."""
    if "prompt" in row and row["prompt"]:
        return row["prompt"]
    lang = row.get("language", "english")
    header = LANGUAGE_PROMPTS.get(lang, LANGUAGE_PROMPTS["english"])
    options = row["options"]
    option_lines = [f"{k}. {v}" for k, v in options.items() if v]
    return (
        f"{header}\n\n"
        f"Question:\n{row['question']}\n\n"
        f"Options:\n" + "\n".join(option_lines) + "\n\nAnswer:"
    )


def pick_dtype():
    if not torch.cuda.is_available():
        return torch.float32
    return torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16


def load_rows(path, split=None):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    if split is not None:
        rows = [r for r in rows if r.get("split") == split]
    for r in rows:
        gold = (
            r.get("gold_answer")
            or r.get("answer")
            or r.get("label")
        )
        if gold is None:
            raise ValueError(f"Riadok {r.get('id')} nemá gold answer.")
        r["_gold"] = str(gold).strip().upper()[:1]
    return rows


def eval_model(model_name, adapter_path, rows, dtype, letters_ids, tokenizer):
    """Vráti dict {summary, preds}."""
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"ADAPTER: {adapter_path if adapter_path else 'NONE (baseline)'}")
    print(f"{'='*70}")

    # Qwen = klasický textový model; Ministral3 = multimodálny (Vision2Seq-like)
    is_ministral = "ministral" in model_name.lower()
    if is_ministral:
        model = Mistral3ForConditionalGeneration.from_pretrained(
            model_name, dtype=dtype, device_map="auto",
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=dtype, device_map="auto",
        )

    if adapter_path is not None:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, adapter_path)
        print("LoRA adapter načítaný.")

    # Zjednoť dtype – base môže byť bf16 (FP8 dequant), LoRA fp16
    base_dtype = next(model.parameters()).dtype
    print(f"Unifikujem dtype na: {base_dtype}")
    model = model.to(base_dtype)

    model.eval()
    device = model.device

    res = []
    stat = collections.defaultdict(lambda: [0, 0])

    for i, r in enumerate(rows, 1):
        prompt = build_prompt_from_row(r)
        text = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True,
        )
        x = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)

        with torch.no_grad():
            logits = model(**x).logits[0, -1]

        n = int(r.get("num_choices", len(r.get("options", {})) or 4))
        pred = LETTERS[int(torch.argmax(logits[letters_ids[:n]]))]
        gold = r["_gold"]
        ok = (pred == gold)

        for k in ("ALL", r.get("language", "unknown"), r.get("source", "unknown")):
            stat[k][0] += int(ok)
            stat[k][1] += 1

        res.append({
            "id": r.get("id"),
            "language": r.get("language"),
            "source": r.get("source"),
            "pred": pred,
            "gold": gold,
            "ok": ok,
        })

        if i % 50 == 0 or i == len(rows):
            print(f"  {i}/{len(rows)}")

    summary = {k: {"acc": round(c / n, 4), "correct": c, "n": n}
               for k, (c, n) in sorted(stat.items())}

    # Uvoľni pamäť pred ďalším modelom
    del model
    torch.cuda.empty_cache()

    return {"summary": summary, "preds": res}


# ============================================================
# MAIN
# ============================================================
def main():
    dtype = pick_dtype()
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
    print("dtype:", dtype)

    rows = load_rows(CONFIG["test_data"], split=CONFIG["split"])
    print(f"Test examples: {len(rows)}")

    # Tokenizer pre písmená – pre oba modely rovnaký koncept
    qwen_tok = AutoTokenizer.from_pretrained(CONFIG["qwen_base"])
    ministral_tok = AutoTokenizer.from_pretrained(CONFIG["ministral_base"])
    for tok in (qwen_tok, ministral_tok):
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token

    qwen_ids = [qwen_tok.encode(l, add_special_tokens=False)[0] for l in LETTERS]
    ministral_ids = [ministral_tok.encode(l, add_special_tokens=False)[0] for l in LETTERS]

    all_results = {}

    # ---- Qwen ----
    all_results["qwen"] = eval_model(
        CONFIG["qwen_base"], CONFIG["qwen_adapter"],
        rows, dtype, qwen_ids, qwen_tok,
    )

    # ---- Ministral ----
    all_results["ministral"] = eval_model(
        CONFIG["ministral_base"], CONFIG["ministral_adapter"],
        rows, dtype, ministral_ids, ministral_tok,
    )

    # ---- Ulož jednotlivé JSONy ----
    qwen_path = os.path.join(CONFIG["out_dir"], "eval_qwen.json")
    ministral_path = os.path.join(CONFIG["out_dir"], "eval_ministral.json")
    json.dump(all_results["qwen"], open(qwen_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)
    json.dump(all_results["ministral"], open(ministral_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

    # ---- Porovnávacia tabuľka ----
    comparison = {}
    for key in all_results["qwen"]["summary"]:
        if key in all_results["ministral"]["summary"]:
            q = all_results["qwen"]["summary"][key]
            m = all_results["ministral"]["summary"][key]
            comparison[key] = {
                "n": q["n"],
                "qwen_acc": q["acc"],
                "ministral_acc": m["acc"],
                "diff_ministral_minus_qwen": round(m["acc"] - q["acc"], 4),
                "winner": "qwen" if q["acc"] > m["acc"] else ("ministral" if m["acc"] > q["acc"] else "tie"),
            }

    comparison_path = os.path.join(CONFIG["out_dir"], "comparison.json")
    json.dump(comparison, open(comparison_path, "w", encoding="utf-8"), indent=2, ensure_ascii=False)

    # ---- Print súhrn ----
    print("\n" + "=" * 80)
    print("POROVNANIE: QWEN vs MINISTRAL")
    print("=" * 80)
    print(f"{'key':<28} {'n':>4} {'qwen':>8} {'ministral':>10} {'diff':>8} {'winner':>10}")
    print("-" * 80)
    for k, v in comparison.items():
        print(f"{k:<28} {v['n']:>4} {v['qwen_acc']:>8.4f} {v['ministral_acc']:>10.4f} "
              f"{v['diff_ministral_minus_qwen']:>+8.4f} {v['winner']:>10}")

    print(f"\nUložené:")
    print(f"  {qwen_path}")
    print(f"  {ministral_path}")
    print(f"  {comparison_path}")


if __name__ == "__main__":
    main()

GPU: NVIDIA A100-SXM4-80GB
dtype: torch.bfloat16
Test examples: 800


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.67GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/458 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/131 [00:00<?, ?B/s]

LoRA adapter načítaný.
Unifikujem dtype na: torch.bfloat16
  50/800
  100/800
  150/800
  200/800
  250/800
  300/800
  350/800
  400/800
  450/800
  500/800
  550/800
  600/800
  650/800
  700/800
  750/800
  800/800

POROVNANIE: QWEN vs MINISTRAL
key                             n     qwen  ministral     diff     winner
--------------------------------------------------------------------------------
ALL                           800   0.5725     0.6238  +0.0513  ministral
arabic                        200   0.6600     0.7250  +0.0650  ministral
arabic_task1_final_test_public  200   0.6600     0.7250  +0.0650  ministral
chinese                       200   0.5200     0.4750  -0.0450       qwen
chinese_task1_final_test_public  200   0.5200     0.4750  -0.0450       qwen
english                       200   0.5650     0.6100  +0.0450  ministral
english_task1_final_test_public  200   0.5650     0.6100  +0.0450  ministral
hindi                         200   0.5450     0.6850  +0.1400  minist